# 1. Configuration et Installation

In [ ]:
# Commande magique Jupyter pour installer silencieusement via pip toutes les librairies nécessaires
!pip install dagshub mlflow transformers tensorflow evaluate scikit-learn pandas emoji


# 2. DagsHub & MLflow Init

In [ ]:
# Importation du module dagshub pour lier le notebook au dépôt distant
import dagshub
# Importation de mlflow pour suivre l'historique et les performances des modèles
import mlflow

# Initialisation de la connexion DagsHub avec les identifiants de votre dépôt et activation du mode MLflow
# PATCH WINDOWS : Force l'utilisation de l'encodage UTF-8 lors de l'écriture des fichiers.
# Ceci corrige l'erreur "charmap codec can't encode characters" causée par le nouveau format d'affichage (summary) de Keras 3
import builtins
_original_open = builtins.open
def _utf8_open(*args, **kwargs):
    mode = kwargs.get('mode', args[1] if len(args) > 1 else 'r')
    if 'b' not in mode and 'encoding' not in kwargs:
        kwargs['encoding'] = 'utf-8'
    return _original_open(*args, **kwargs)
builtins.open = _utf8_open

dagshub.init(repo_owner='Oscar-AS', repo_name='disaster-tweets-project', mlflow=True)

# Définition du nom du dossier (expérience) dans MLflow où toutes nos métriques seront classées
mlflow.set_experiment("Disaster_Tweets_Niveau_3_et_4")

# Affichage d'un message console pour confirmer que le tracking est bien connecté
print("MLflow activé avec succès sur DagsHub !")


# 3. Importation et Prétraitement des Données
Ici nous allons gérer la **traduction des emojis** et nettoyer le bruit (URLs).

In [ ]:
# Importation de pandas pour la gestion des données tabulaires (DataFrame)
import pandas as pd
# Importation de 're' pour utiliser les expressions régulières (Regex) lors du nettoyage de texte
import re
# Importation de la librairie emoji pour convertir les symboles visuels en texte compréhensible
import emoji
# Importation de train_test_split pour diviser notre jeu de données (Train et Test)
from sklearn.model_selection import train_test_split

# 1. Chargement des données
# Lecture du fichier CSV depuis le dossier Base et stockage dans la variable 'df'
df = pd.read_csv("Base/tweets.csv")

# 2. Fonction de nettoyage "Léger" pour Deep Learning / Transformers
# Définition de la fonction personnalisée de nettoyage
def clean_text_advanced(text):
    # Traduction de chaque emoji en texte (ex: un camion de pompier devient ':fire_engine:')
    text = emoji.demojize(text)
    
    # Remplacement des deux points ":" ajoutés par demojize par un espace classique
    text = text.replace(":", " ")
    # Remplacement des tirets bas "_" ajoutés par demojize par un espace classique
    text = text.replace("_", " ")
    
    # Suppression complète de toutes les URLs (commençant par http, https ou www)
    text = re.sub(r"http\S+|www\S+|https\S+", "", text, flags=re.MULTILINE)
    
    # Remplacement des mentions utilisateur (ex: @jean) par un "token" générique [USER]
    text = re.sub(r"\@\w+", "[USER]", text)
    
    # Remplacement de plusieurs espaces consécutifs par un seul espace unique
    text = re.sub(r"\s+", " ", text)
    # Retrait des espaces restants au tout début ou à la toute fin de la phrase
    text = text.strip()
    
    # Retourne la phrase nettoyée
    return text

# Application de la fonction de nettoyage sur toute la colonne 'text' pour créer la nouvelle colonne 'clean_text'
df['clean_text'] = df['text'].apply(clean_text_advanced)

# 3. Séparation Train/Test (80% / 20%)
# Séparation des textes (X) et des labels (y) en gardant 20% pour l'évaluation. 
# "stratify=df['target']" garantit que l'on garde 19% de désastres dans les DEUX sets.
X_train, X_test, y_train, y_test = train_test_split(
    df['clean_text'], df['target'], test_size=0.2, random_state=42, stratify=df['target']
)

# Affiche dans la console le nombre de tweets utilisés pour l'entraînement
print(f"Taille de l'entraînement : {len(X_train)}")
# Affiche dans la console le nombre de tweets gardés pour le test
print(f"Taille du test : {len(X_test)}")


# 4. Modèles de Niveau 3 (Réseaux de Neurones avec TensorFlow/Keras)

In [ ]:
# Importation de TensorFlow, la librairie Deep Learning principale de Google
import tensorflow as tf
# Importation spécifique de la couche de vectorisation de texte de Keras
from tensorflow.keras.layers import TextVectorization
# Importation de NumPy pour la manipulation avancée des tableaux mathématiques
import numpy as np

# Définition de la taille maximale du vocabulaire autorisé (les 15000 mots les plus fréquents)
MAX_VOCAB_SIZE = 15000
# Définition de la taille maximale d'une phrase (tronquée si plus longue, remplie par des 0 si plus courte)
MAX_SEQUENCE_LENGTH = 128

# Instanciation de la couche de Vectorisation
vectorizer = TextVectorization(
    max_tokens=MAX_VOCAB_SIZE, # Limite du vocabulaire
    output_mode='int',         # Chaque mot sera remplacé par un nombre entier
    output_sequence_length=MAX_SEQUENCE_LENGTH # Fixe la longueur de toutes les séquences à 128
)

# Apprentissage du vocabulaire : on lit le texte d'entraînement pour créer le dictionnaire mot -> entier
vectorizer.adapt(X_train.to_numpy())

# Fonction utilitaire pour préparer les données afin que TensorFlow s'entraîne plus vite
def prepare_tf_dataset(X, y, batch_size=32):
    # Création d'un dataset TensorFlow à partir de nos listes Python (X et y)
    dataset = tf.data.Dataset.from_tensor_slices((X, y))
    # Groupement des données en paquets (batches) de 32, et mise en mémoire cache dynamique (AUTOTUNE)
    dataset = dataset.batch(batch_size).prefetch(tf.data.AUTOTUNE)
    # Retourne le dataset optimisé
    return dataset

# Création du Dataset d'entraînement accéléré
train_ds = prepare_tf_dataset(X_train, y_train)
# Création du Dataset de test accéléré
test_ds = prepare_tf_dataset(X_test, y_test)

# Importation du module MLflow dédié à TensorFlow
import mlflow.tensorflow
# Activation du suivi automatique (enregistrera la loss, les paramètres et les modèles à chaque epoch sans coder manuellement)
mlflow.tensorflow.autolog(log_models=True)


## Modèle 3.1 : LSTM (Long Short-Term Memory) Simple

### Description du modèle
Le LSTM est l'architecture classique des réseaux de neurones pour le traitement de texte. Il appartient à la famille des Réseaux de Neurones Récurrents (RNN).

### Explication du fonctionnement
Contrairement aux réseaux classiques qui traitent l'information d'un bloc, le LSTM lit le texte **mot par mot, de gauche à droite**. Il possède une "mémoire interne" (une bande transporteuse mathématique) qui lui permet de se souvenir du début de la phrase quand il arrive à la fin. Cela lui permet de comprendre le contexte.

### Pourquoi l'utiliser ?
C'est le modèle d'introduction parfait pour le Deep Learning sur du texte. Il saisit mieux le sens global qu'un simple modèle ML classique, mais reste rapide à entraîner.


In [ ]:
# Importation des types de couches (layers) et de l'architecture séquentielle (models) de Keras
from tensorflow.keras import layers, models
# Importation de la fonction générant un rapport complet de classification (F1, Precision, Recall, etc.)
from sklearn.metrics import classification_report, f1_score, precision_score, recall_score, accuracy_score, fbeta_score

# Démarrage manuel d'une instance (run) MLflow nommée "3.1_LSTM_Simple"
with mlflow.start_run(run_name="3.1_LSTM_Simple"):
    # Initialisation d'un réseau de neurones en couches successives (Sequential)
    model_lstm = models.Sequential([
        # Couche 1 : Vectorisation (transforme le texte brut en liste d'entiers)
        vectorizer,
        # Couche 2 : Embedding (transforme l'entier en un vecteur mathématique dense de dimension 64)
        layers.Embedding(input_dim=MAX_VOCAB_SIZE, output_dim=64, mask_zero=True),
        # Couche 3 : LSTM (le coeur récurrent, avec 64 neurones)
        layers.LSTM(64),
        # Couche 4 : Couche finale (Dense) avec 1 neurone et une activation Sigmoid (renvoie une probabilité entre 0 et 1)
        layers.Dense(1, activation='sigmoid')
    ])
    
    # Compilation du modèle : définition de la fonction d'erreur (binary_crossentropy) et de l'optimiseur (adam)
    model_lstm.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy'])
    # Lancement de l'entraînement sur 3 itérations (epochs), en validant sur test_ds à chaque fin d'epoch
    model_lstm.fit(train_ds, validation_data=test_ds, epochs=3)
    
    # Prédiction sur le jeu de test. Si la probabilité est > 0.5, on classe 1 (Désastre), sinon 0 (astye(int) convertit False/True en 0/1)
    y_pred_lstm = (model_lstm.predict(test_ds) > 0.5).astype(int)
    
    # Affichage du titre du rapport
    print("\n--- Rapport LSTM Simple ---")
    # Affichage des résultats complets en confrontant les vraies valeurs (y_test) aux prédictions (y_pred_lstm)
    print(classification_report(y_test, y_pred_lstm))
    
    # Suivi explicite des métriques de test dans MLflow
    mlflow.log_metric("eval_f1_macro", f1_score(y_test, y_pred_lstm, average='macro'))
    mlflow.log_metric("eval_f2_score", fbeta_score(y_test, y_pred_lstm, beta=2, average='macro'))
    
    precision_cls = precision_score(y_test, y_pred_lstm, average=None)
    recall_cls = recall_score(y_test, y_pred_lstm, average=None)
    mlflow.log_metric("eval_precision_class_0", precision_cls[0])
    mlflow.log_metric("eval_precision_class_1", precision_cls[1])
    mlflow.log_metric("eval_recall_class_0", recall_cls[0])
    mlflow.log_metric("eval_recall_class_1", recall_cls[1])
    mlflow.log_metric("eval_accuracy", accuracy_score(y_test, y_pred_lstm))
    
    # Enregistrement explicite du modèle pour la mise en production
    try:
        mlflow.tensorflow.log_model(model_lstm, "model")
    except Exception as e:
        print("Erreur lors de la sauvegarde du modèle Keras :", e)


## Modèle 3.2 : TextCNN (Convolutional Neural Network 1D)

### Description du modèle
Historiquement inventé pour l'analyse d'images (pour détecter des bords, des formes), le CNN a été adapté pour le texte (TextCNN) avec un succès retentissant.

### Explication du fonctionnement
Au lieu de lire mot par mot, le CNN utilise des "fenêtres glissantes" (Filtres Convolutifs) qui regardent des groupes de 3, 4 ou 5 mots à la fois. Le modèle cherche spécifiquement des **"motifs" ou "expressions clés"** (ex: "building on fire", "heavy earthquake") indépendamment de leur position dans la phrase.

### Pourquoi l'utiliser ?
Pour des textes très courts comme des **tweets**, le TextCNN est souvent plus performant et surtout **plus rapide** à s'entraîner que le LSTM, car il ne s'encombre pas d'une mémoire complexe à long terme.


In [ ]:
# Démarrage d'un nouveau Run MLflow nommé "3.2_TextCNN"
with mlflow.start_run(run_name="3.2_TextCNN"):
    # Création d'un nouveau modèle en couches empilées
    model_cnn = models.Sequential([
        # Couche 1 : Vectorisation (texte vers entiers)
        vectorizer,
        # Couche 2 : Embedding (entiers vers vecteurs mathématiques)
        layers.Embedding(input_dim=MAX_VOCAB_SIZE, output_dim=64),
        # Couche 3 : Convolution 1D, qui utilise 64 filtres et lit des paquets de 5 mots (kernel_size=5) avec une fonction d'activation ReLU
        layers.Conv1D(filters=64, kernel_size=5, activation='relu'),
        # Couche 4 : Regroupement Max Global (ne garde que l'information la plus importante détectée par la convolution)
        layers.GlobalMaxPooling1D(),
        # Couche 5 : Couche dense de 32 neurones pour interpréter l'information
        layers.Dense(32, activation='relu'),
        # Couche 6 : Dropout (désactive aléatoirement 50% des neurones pour éviter le surapprentissage)
        layers.Dropout(0.5),
        # Couche 7 : Prédiction binaire finale (1 neurone, Sigmoid)
        layers.Dense(1, activation='sigmoid')
    ])
    
    # Préparation du modèle avec l'optimiseur adam et suivi de l'accuracy
    model_cnn.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy'])
    # Entraînement pendant 3 epochs
    model_cnn.fit(train_ds, validation_data=test_ds, epochs=3)
    
    # Récupération des prédictions formatées en 0 ou 1
    y_pred_cnn = (model_cnn.predict(test_ds) > 0.5).astype(int)
    
    # Affichage dans la console
    print("\n--- Rapport TextCNN ---")
    # Impression du rapport final (F1, Precision, Recall)
    print(classification_report(y_test, y_pred_cnn))
    
    # Suivi explicite des métriques de test dans MLflow
    mlflow.log_metric("eval_f1_macro", f1_score(y_test, y_pred_cnn, average='macro'))
    mlflow.log_metric("eval_f2_score", fbeta_score(y_test, y_pred_cnn, beta=2, average='macro'))
    
    precision_cls = precision_score(y_test, y_pred_cnn, average=None)
    recall_cls = recall_score(y_test, y_pred_cnn, average=None)
    mlflow.log_metric("eval_precision_class_0", precision_cls[0])
    mlflow.log_metric("eval_precision_class_1", precision_cls[1])
    mlflow.log_metric("eval_recall_class_0", recall_cls[0])
    mlflow.log_metric("eval_recall_class_1", recall_cls[1])
    mlflow.log_metric("eval_accuracy", accuracy_score(y_test, y_pred_cnn))
    
    # Enregistrement explicite du modèle pour la mise en production
    try:
        mlflow.tensorflow.log_model(model_cnn, "model")
    except Exception as e:
        print("Erreur lors de la sauvegarde du modèle Keras :", e)


## Modèle 3.3 : BiLSTM + GRU avec Poids de Classe (Le Modèle Ultime Niveau 3)

### Description du modèle
C'est une version sous stéroïdes du LSTM, combinant plusieurs architectures récurrentes.

### Explication du fonctionnement
1. **BiDirectional :** Au lieu de lire la phrase uniquement de gauche à droite, la couche BiLSTM la lit *aussi* de droite à gauche en même temps. Cela permet de comprendre qu'un mot a été influencé par la fin de la phrase.
2. **Couche GRU :** Un parent plus moderne et plus léger du LSTM, ajouté ici pour extraire une dernière série d'informations.
3. **Class Weights :** Nous modifions l'équation d'entraînement pour que le modèle soit puni sévèrement s'il rate un tweet de "Désastre" (classe 1).

### Pourquoi l'utiliser ?
C'est souvent le maximum de performance que l'on peut atteindre avant de passer aux immenses Transformers.


In [ ]:
# Calcul mathématique pour compenser le déséquilibre des classes
# Récupération du nombre total d'exemples d'entraînement
total = len(y_train)
# Comptage du nombre d'exemples positifs (les 1, donc les vrais désastres)
pos = sum(y_train)
# Comptage du nombre d'exemples négatifs (les 0)
neg = total - pos

# Poids pour la classe 0 : un petit nombre (car la classe est majoritaire)
weight_for_0 = (1 / neg) * (total / 2.0)
# Poids pour la classe 1 : un grand nombre (car la classe est minoritaire)
weight_for_1 = (1 / pos) * (total / 2.0)
# Dictionnaire regroupant ces poids pour TensorFlow
class_weights = {0: weight_for_0, 1: weight_for_1}

# Lancement du Run MLflow pour le modèle optimisé
with mlflow.start_run(run_name="3.3_BiLSTM_Optimized"):
    # Architecture avancée du réseau
    model_opt = models.Sequential([
        # Vectorisation
        vectorizer,
        # Embedding plus puissant (128 dimensions au lieu de 64). mask_zero=True ignore le padding vide.
        layers.Embedding(input_dim=MAX_VOCAB_SIZE, output_dim=128, mask_zero=True),
        # La couche Bidirectional enveloppe le LSTM. 'return_sequences=True' permet d'enchaîner avec le GRU
        layers.Bidirectional(layers.LSTM(64, return_sequences=True)),
        # Dropout de 30% pour forcer le modèle à généraliser
        layers.Dropout(0.3),
        # Couche GRU, qui est un RNN plus rapide
        layers.GRU(32),
        # Une couche Dense classique pour extraire les conclusions
        layers.Dense(32, activation='relu'),
        # Un dernier Dropout avant la sortie
        layers.Dropout(0.3),
        # Sortie binaire
        layers.Dense(1, activation='sigmoid')
    ])
    
    # Utilisation d'un "Learning Rate" (taux d'apprentissage) personnalisé, plus faible que celui par défaut
    optimizer = tf.keras.optimizers.Adam(learning_rate=1e-4)
    # Compilation
    model_opt.compile(loss='binary_crossentropy', optimizer=optimizer, metrics=['accuracy'])
    # Entraînement sur 5 epochs, EN INCLUANT l'argument 'class_weight' défini plus haut
    model_opt.fit(train_ds, validation_data=test_ds, epochs=5, class_weight=class_weights)
    
    # Transformation des probabilités en entiers 0/1
    y_pred_opt = (model_opt.predict(test_ds) > 0.5).astype(int)
    
    # Affichage
    print("\n--- Rapport BiLSTM Optimisé ---")
    print(classification_report(y_test, y_pred_opt))
    
    # Suivi explicite des métriques de test dans MLflow
    mlflow.log_metric("eval_f1_macro", f1_score(y_test, y_pred_opt, average='macro'))
    mlflow.log_metric("eval_f2_score", fbeta_score(y_test, y_pred_opt, beta=2, average='macro'))
    
    precision_cls = precision_score(y_test, y_pred_opt, average=None)
    recall_cls = recall_score(y_test, y_pred_opt, average=None)
    mlflow.log_metric("eval_precision_class_0", precision_cls[0])
    mlflow.log_metric("eval_precision_class_1", precision_cls[1])
    mlflow.log_metric("eval_recall_class_0", recall_cls[0])
    mlflow.log_metric("eval_recall_class_1", recall_cls[1])
    mlflow.log_metric("eval_accuracy", accuracy_score(y_test, y_pred_opt))
    
    # Enregistrement explicite du modèle pour la mise en production
    try:
        mlflow.tensorflow.log_model(model_opt, "model")
    except Exception as e:
        print("Erreur lors de la sauvegarde du modèle Keras :", e)


# 5. Modèles de Niveau 4 (Transformers via Hugging Face)
On entre dans l'état de l'art du NLP.

In [ ]:
# Import de l'objet Dataset de Hugging Face
from datasets import Dataset
# Import de evaluate pour calculer de façon standardisée les métriques d'évaluation
import evaluate
# Import de NumPy
import numpy as np

# Transformation de notre DataFrame d'entraînement Pandas en un objet "Dataset" ultra-optimisé de Hugging Face
hf_train = Dataset.from_pandas(pd.DataFrame({'text': X_train, 'label': y_train}))
# Transformation de notre DataFrame de test Pandas
hf_test = Dataset.from_pandas(pd.DataFrame({'text': X_test, 'label': y_test}))

# Importation de scikit-learn pour calculer facilement le F2-Score et les métriques par classe
from sklearn.metrics import f1_score, precision_score, recall_score, accuracy_score, fbeta_score

# Fonction exécutée à la fin de chaque Epoch par le Trainer pour calculer le score
def compute_metrics(eval_pred):
    # Séparation des probabilités prédites (logits) et des vraies réponses (labels)
    logits, labels = eval_pred
    # L'argmax récupère la classe ayant reçu la plus forte probabilité (0 ou 1)
    predictions = np.argmax(logits, axis=-1)
    
    # Précision et Rappel par classe
    precision_cls = precision_score(labels, predictions, average=None)
    recall_cls = recall_score(labels, predictions, average=None)
    
    # Calcul des métriques globales
    f1 = f1_score(labels, predictions, average="macro")
    f2 = fbeta_score(labels, predictions, beta=2, average="macro")
    accuracy = accuracy_score(labels, predictions)
    
    # Retourner toutes les métriques pour le suivi MLflow (le Trainer ajoutera automatiquement le préfixe "eval_")
    return {
        "f1_macro": f1,
        "f2_score": f2,
        "precision_class_0": precision_cls[0],
        "precision_class_1": precision_cls[1],
        "recall_class_0": recall_cls[0],
        "recall_class_1": recall_cls[1],
        "accuracy": accuracy
    }

# Importation du système d'exploitation
import os
# Paramétrage de la variable d'environnement qui indique à Hugging Face dans quel dossier MLflow il doit écrire
os.environ["MLFLOW_EXPERIMENT_NAME"] = "Disaster_Tweets_Niveau_3_et_4"


In [ ]:
# Import des AutoClasses (la magie de Hugging Face pour importer n'importe quel modèle du web en 1 ligne)
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
import torch

# Définition d'une fonction Python réutilisable pour entraîner n'importe quel Transformer sans réécrire le code
def train_hf_model(model_id, run_name, batch_size=16, epochs=2):
    # Affichage du démarrage
    print(f"========== Début de l'entraînement pour {model_id} ==========")
    
    # 1. Chargement du Tokenizer spécifique au modèle (le dictionnaire qui transforme les mots en IDs)
    tokenizer = AutoTokenizer.from_pretrained(model_id)
    
    # Fonction qui applique le tokenizer sur une phrase
    def tokenize_function(examples):
        # On coupe les phrases (truncation=True) à 128 "tokens" maximum et on ajoute du vide (padding) pour les plus courtes
        return tokenizer(examples['text'], padding="max_length", truncation=True, max_length=128)
    
    # Application massive et extrêmement rapide (batched=True) du Tokenizer sur tout le jeu d'entraînement
    tokenized_train = hf_train.map(tokenize_function, batched=True)
    # Même chose pour le test
    tokenized_test = hf_test.map(tokenize_function, batched=True)
    
    # 2. Chargement de l'architecture du Transformer avec une tête de classification pour 2 sorties (0 ou 1)
    model = AutoModelForSequenceClassification.from_pretrained(model_id, num_labels=2)
    
    # 3. Paramètres de l'entraînement 
    training_args = TrainingArguments(
        output_dir=f"./results_{run_name}",  # Dossier de sauvegarde
        evaluation_strategy="epoch",         # Evaluer le modèle à la fin de chaque Epoch
        save_strategy="epoch",               # Sauvegarder un "point de contrôle" à la fin de chaque Epoch
        learning_rate=2e-5,                  # Taux d'apprentissage très petit (spécifique aux Transformers)
        per_device_train_batch_size=batch_size, # Taille des paquets envoyés à la carte graphique (entraînement)
        per_device_eval_batch_size=batch_size,  # Taille des paquets envoyés à la carte graphique (test)
        num_train_epochs=epochs,             # Nombre total d'itérations
        weight_decay=0.01,                   # Ajout de pénalités pour éviter le surapprentissage
        load_best_model_at_end=True,         # A la fin, on recharge la version qui a eu le meilleur score
        report_to="mlflow",                  # Dit au système d'envoyer tout le suivi de cet entraînement vers MLflow (DagsHub)
        run_name=run_name,                   # Nom du run dans l'interface MLflow
    )
    
    # 4. L'Objet Trainer qui s'occupe de gérer toute la boucle mathématique PyTorch en arrière-plan
    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=tokenized_train,
        eval_dataset=tokenized_test,
        compute_metrics=compute_metrics, # On utilise notre fonction personnalisée pour mesurer le F1-Score
    )
    
    # Démarre l'entraînement intensif
    trainer.train()
    
    try:
        # Tente d'enregistrer le modèle HuggingFace dans MLflow pour la mise en production
        components = {"model": trainer.model, "tokenizer": tokenizer}
        mlflow.transformers.log_model(transformers_model=components, artifact_path="model")
    except Exception as e:
        print("Avertissement: L'enregistrement du modèle Transformers dans MLflow a échoué:", e)
    
    # Force MLflow à fermer proprement la session de suivi de ce run
    mlflow.end_run()
    # Affiche la fin dans la console
    print(f"========== Fin de l'entraînement pour {model_id} ==========\n")


## Modèle 4.1 : DistilBERT

### Description du modèle
DistilBERT est une version "compressée" du modèle BERT original de Google.

### Explication du fonctionnement
Il utilise une technique appelée **Distillation de Connaissance** : on prend un énorme modèle (BERT) qui sert de "Professeur", et on entraîne un petit modèle ("l'Élève", DistilBERT) à reproduire les réponses du professeur. Le résultat est un modèle 40% plus petit, 60% plus rapide, mais qui conserve 97% des capacités de compréhension du langage de BERT.

### Pourquoi l'utiliser ?
C'est le modèle parfait pour faire du prototypage rapide. Il donne d'excellents résultats en un temps record.


In [ ]:
# Appel de notre super-fonction pour entraîner DistilBERT. 
# Comme il est très rapide, on peut s'autoriser de faire 3 epochs au lieu de 2.
train_hf_model(model_id="distilbert-base-uncased", run_name="4.1_DistilBERT", epochs=3)


## Modèle 4.2 : BERT (Bidirectional Encoder Representations from Transformers)

### Description du modèle
Le modèle révolutionnaire publié par Google en 2018 qui a changé l'histoire du NLP.

### Explication du fonctionnement
Contrairement aux anciens modèles qui lisaient le texte de gauche à droite, BERT utilise le **Mécanisme d'Attention**. Il regarde *tous* les mots de la phrase simultanément pour comprendre la relation exacte de chaque mot par rapport à tous les autres mots. (Le fameux "Contexte Bidirectionnel Profond"). Il a été pré-entraîné en lisant tout Wikipédia.

### Pourquoi l'utiliser ?
C'est la référence absolue. Très lourd, mais d'une précision redoutable pour comprendre le double sens des mots.


In [ ]:
# Appel de la fonction pour entraîner le modèle BERT originel, version uncased (tout en minuscules)
train_hf_model(model_id="bert-base-uncased", run_name="4.2_BERT_Classique", epochs=2)


## Modèle 4.3 : RoBERTa (Robustly Optimized BERT)

### Description du modèle
C'est la version optimisée de BERT, créée par l'équipe d'Intelligence Artificielle de Facebook/Meta.

### Explication du fonctionnement
RoBERTa possède exactement la même architecture que BERT, mais Meta a découvert que Google avait "sous-entraîné" BERT. RoBERTa a été entraîné beaucoup plus longtemps, sur un volume de texte bien plus massif (incluant des flux de réseaux sociaux et Reddit), et en supprimant certaines tâches d'apprentissage jugées inutiles (comme la prédiction de la phrase suivante).

### Pourquoi l'utiliser ?
Sur des données informelles et du langage web (comme Twitter), **RoBERTa est presque systématiquement supérieur à BERT**.


In [ ]:
# Appel de la fonction pour entraîner RoBERTa
train_hf_model(model_id="roberta-base", run_name="4.3_RoBERTa", epochs=2)


## Modèle 4.4 : DeBERTa-v3 (L'État de l'Art Suprême)

### Description du modèle
Créé par Microsoft, c'est l'un des meilleurs modèles au monde aujourd'hui pour les tâches de classification.

### Explication du fonctionnement
DeBERTa améliore le mécanisme d'attention en utilisant une **Attention Désintriquée (Disentangled Attention)**. Là où BERT mélange le "mot" et sa "position absolue dans la phrase", DeBERTa traite le contenu du mot et sa position relative séparément. De plus, la version v3 utilise une technique spéciale d'apprentissage génératif qui le rend extrêmement robuste.

### Pourquoi l'utiliser ?
Si vous participez à une compétition Kaggle, **c'est LE modèle à utiliser**. Il gagne 90% des compétitions de classification.


In [ ]:
# Entraînement de DeBERTa.
# IMPORTANT : J'ai passé l'argument batch_size=8 car ce modèle consomme beaucoup plus de mémoire vidéo (VRAM).
# Sans cette réduction, l'entraînement risque de faire crasher la carte graphique (Erreur OOM : Out Of Memory).
train_hf_model(model_id="microsoft/deberta-v3-base", run_name="4.4_DeBERTa", batch_size=8, epochs=2)


# 6. Sélection du Meilleur Modèle (Mise en Production)
Nous interrogeons l'historique MLflow pour trouver le meilleur modèle selon le **F2-Score** et le **Rappel (Classe 1)**, puis nous l'enregistrons dans le Model Registry.

In [ ]:
import mlflow
from mlflow.tracking import MlflowClient

# 1. Obtenir l'ID de notre expérience
experiment = mlflow.get_experiment_by_name("Disaster_Tweets_Niveau_3_et_4")

if experiment is not None:
    # 2. Récupérer tous les runs de cette expérience
    runs = mlflow.search_runs(experiment_ids=[experiment.experiment_id])
    
    # On s'assure d'avoir des résultats
    if not runs.empty and 'metrics.eval_f2_score' in runs.columns:
        # 3. Trier les modèles. 
        # Critère 1 : Le plus haut F2-Score (qui privilégie le rappel)
        # Critère 2 : Le plus haut Rappel sur la classe 1 (les vrais désastres) en cas d'égalité
        sorted_runs = runs.sort_values(
            by=['metrics.eval_f2_score', 'metrics.eval_recall_class_1'], 
            ascending=[False, False]
        )
        
        # 4. Prendre le meilleur run
        best_run = sorted_runs.iloc[0]
        best_run_id = best_run['run_id']
        best_run_name = best_run['tags.mlflow.runName']
        best_f2 = best_run['metrics.eval_f2_score']
        best_recall = best_run['metrics.eval_recall_class_1']
        
        print("=== MEILLEUR MODÈLE TROUVÉ ===")
        print(f"Nom du Run : {best_run_name}")
        print(f"F2-Score : {best_f2:.4f}")
        print(f"Recall (Désastres) : {best_recall:.4f}")
        print(f"Run ID : {best_run_id}")
        print("==============================")
        
        # 5. Enregistrer ce modèle dans le Model Registry MLflow
        model_uri = f"runs:/{best_run_id}/model"
        model_name = "Disaster_Tweet_Predictor_Prod"
        
        try:
            print(f"\nEnregistrement du modèle '{model_name}' dans le MLflow Registry...")
            # Enregistrement
            registered_model = mlflow.register_model(model_uri=model_uri, name=model_name)
            
            # 6. Passer le modèle en phase de "Production"
            client = MlflowClient()
            client.transition_model_version_stage(
                name=model_name,
                version=registered_model.version,
                stage="Production",
                archive_existing_versions=True
            )
            print(f"Succès ! Le modèle version {registered_model.version} est maintenant en PRODUCTION.")
            print(f"Vous pouvez le charger plus tard avec : mlflow.pyfunc.load_model(f'models:/{model_name}/Production')")
            
        except Exception as e:
            print("Erreur lors de l'enregistrement dans le Registry. Si vous êtes en local sans serveur de tracking avancé, c'est normal.")
            print("Détail de l'erreur :", e)
    else:
        print("Aucune métrique 'eval_f2_score' trouvée. Assurez-vous d'avoir entraîné les modèles d'abord.")
else:
    print("L'expérience MLflow n'a pas été trouvée.")
